<a href="https://www.kaggle.com/code/nadijfer/spam-email-using-ann-from-scratch?scriptVersionId=288839977" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# **Detecting Spam Email using Simple Neural Network from Scratch**

# I. Introduction

This notebook is my attempt on implementing what I've learnt in class, for have taken natural language processing (NLP) and artificial neural network (ANN). It aims to test my understanding both mathematically and in code implementation by building from scratch without any modules, except modules below. References primarily used from lecture notes and class' presentations. For this notebook, I tried to write codes and its description as clear as possible, by adding docstrings to functions.

The notebook will be separated into two implementation: using vanilla Bag-of-Words (BoW) for baseline, TF-IDF and  Word Embeddings as its improvement.

# Baseline Implementation using BoW

In [1]:
import numpy as np
import pandas as pd
import string
import time

# II. Exploratory Data Analysis (EDA)

In [2]:
df = pd.read_csv('../data/raw/emails.csv')
df

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
...,...,...
5723,Subject: re : research and development charges...,0
5724,"Subject: re : receipts from visit jim , than...",0
5725,Subject: re : enron case study update wow ! a...,0
5726,"Subject: re : interest david , please , call...",0


### Null or NaN values

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5728 non-null   object
 1   spam    5728 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 89.6+ KB


No null or NaN value with equal distribution of both classes. We're good to go.

# II. Text Processing

## Cleaning and Tokenizing Documents

In [4]:
def preprotext(text):
    """Cleaning text input by applying lowercase, removing punctuation, and tokenizing (text.split()). The function could be used as:

    cleaned_texts = texts.copy()
    for i in range(len(texts)):
        cleaned_texts[i] = preprotext(texts[i])

    Args:
        text: string to be cleaned.

    Returns:
        list: tokens of cleaned text.
    """
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation)) # remove punctuation marks
    tokens = text.split() # tokenizing
    return tokens

## Building Vocabulary

Vocabulary $|V|$ is a list of words model used as dictionary.

In [5]:
def build_vocab(documents):
    """Building list of words from train data for model. The function could be used as:
    vocab = build_vocab(cleaned_texts)
    len(vocab), vocab

    Args:
        documents: list of words

    Returns:
        list: vocabulary
    """
    vocab = []
    for document in documents:
        tokens = preprotext(document) # resulting tokens of each documents
        for token in tokens:
            vocab.append(token)
    
    vocab = list(set(vocab))
    vocab = {word: index for index, word in enumerate(vocab)}
    return vocab

## Encoding Text with Bag-of-Words

In [6]:
def build_bow(cleaned_texts, vocab):
    """Build BoW to a token of clened texts
    doc_bow = [] # bow for each documents
    for docs in cleaned_texts:
        docs = build_bow(docs)
        doc_bow.append(docs)
    """
    bow = np.zeros((len(vocab)))
    for token in cleaned_texts:
        if token in vocab:
            bow[vocab[token]] += 1
    return bow

# IV. Training Neural Network

The architecture of NN is built upon Multilayer Perceptron (MLP) with a hidden layer consisting 128 hidden nodes $H$. For binary target $t$, the loss function $\mathcal L$ used is binary cross entropy (BCE), defined as:

$$
\mathcal{L} = - \sum y \cdot \log (\hat y)
$$

Where $y$ is the true label (we defined in this notebook as `t` for target) and $\hat{y}$ is the prediction label (defined as `y`). 

Activation functions for input-to-hidden layer is ReLU, the function itself and its derivative defined as:

$$
\begin{align*}
f(x) & = \mathrm{ReLU} = \begin{cases}x, & \text{if } x > 0\\ 0, & \text{otherwise}\end{cases}\\
f'(x) & = \begin{cases}1\\ 0\end{cases}\\
\end{align*}
$$

and the activation functions for hidden-to-output layer is Sigmoid, both the function itself and its derivative defined as:

$$
\begin{align*}
f(x) & = \sigma = \frac{1}{1 + e^{-x}} \\
f'(x) & = f(x) (1-f(x))
\end{align*}
$$

Such architecture could be implemented as:

```
Input(len(vocab)) -> Hidden(128; Relu -> Sigmoid) -> y(1) -> L(BCE)
```

In [7]:
N = len(df) # length of all the data
H = 128 # hidden nodes (size), 1 hidden layer
lr = 1e-4 # learning rate

### Activation functions and its derivatives

In [8]:
def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def sigmoid(x):
    return 1/(1+np.exp(-x))

def sigmoid_deriv(x):
    return sigmoid(x) * (1-sigmoid(x))

### Loss function and its derivative (BCE)

In [9]:
def BCE(t, y):
    return - (t * np.log(y))

def BCE_deriv(t, y): # only when sigmoid is used
    return y - t

## Train-test split

In [10]:
X = df["text"]
y = df["spam"]

idx = np.random.permutation(N)

split = int(0.8 * N)

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

## Building Vocabulary, Cleaning Text and Encoding BoW

In [11]:
vocab = build_vocab(X_train)

In [12]:
train_cleaned = X_train.copy()
for i in range(len(X_train)):
    train_cleaned.iloc[i] = preprotext(train_cleaned.iloc[i])

In [13]:
train_bow = []
for tokens in train_cleaned:
    tokens = build_bow(tokens, vocab)
    train_bow.append(tokens)

## Initializing NN parameters

In [14]:
x = np.array(train_bow) # input
t = y_train # target
w = np.random.randn(x.shape[1], H) * np.sqrt(2/len(X_train)) # He initialization
b0 = np.zeros(H) # hidden bias
v = np.random.randn(H, 1) * np.sqrt(1/H) # Xavier initialization
b1 = np.zeros(1) # output bias

In [15]:
# this cell is to check the values before and after training
params = {
    'w': w,
    'b0': b0,
    'v': v,
    'b1': b1
}

params

{'w': array([[ 0.01017145,  0.03681166,  0.02285528, ...,  0.00145732,
         -0.01262457, -0.00658382],
        [ 0.02214439,  0.02673515, -0.01710286, ..., -0.01942932,
          0.02255406, -0.02467912],
        [-0.01427463, -0.00464749, -0.00465425, ..., -0.01333243,
          0.02942933, -0.03812609],
        ...,
        [-0.0289793 ,  0.01513426, -0.02534047, ...,  0.02406434,
         -0.0250462 ,  0.00182167],
        [-0.00558504,  0.0131883 ,  0.00157024, ..., -0.00064532,
         -0.03689956,  0.00985135],
        [-0.00526775, -0.00143216,  0.02459505, ..., -0.0082632 ,
          0.01796811, -0.01003803]], shape=(34609, 128)),
 'b0': array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.

In [16]:
x.shape, w.shape, b0.shape, v.shape, b1.shape

((4582, 34609), (34609, 128), (128,), (128, 1), (1,))

## Backpropagation

Backpropagation to a parameter at input layer $\theta$ could be defined as:

$$
\frac{\partial L}{\partial \theta} = \frac{\partial L}{\partial y_{out}} \frac{\partial y_{out}}{\partial y_{in}} \frac{\partial y_{in}}{\partial z_{out}} \frac{\partial z_{out}}{\partial z_{in}} \frac{\partial z_{in}}{\partial \theta}
$$

Special condition occurs to binary cross entropy (BCE) loss function when is used next to sigmoid activation function. Both BCE and its derivative could be defined as:

$$
\begin{align*}
f(x) = \mathcal{L} &= - \sum^N_{i=1} y \cdot \log(\hat{y}) \\
f'(x) &= \hat{y} - y
\end{align*}
$$

Where $y$ is the true label (we defined in this notebook as `t` for target) and $\hat{y}$ is the prediction label (defined as `y`), the function could be simplified as:

$$
\begin{align*}
\frac{\partial L}{\partial \theta} &= \frac{\partial L}{\partial y_{out}} \frac{\partial y_{out}}{\partial y_{in}} \frac{\partial y_{in}}{\partial z_{out}} \frac{\partial z_{out}}{\partial z_{in}} \frac{\partial z_{in}}{\partial \theta} \\
 & = (\hat{y}-y) \cdot v \cdot \mathrm{ReLU}'(z_{in}) \cdot \Omega
\end{align*}
$$

Where $\Omega$ depends whether observing input biases or weight biases, such as:

$$
\Omega = 
    \begin{cases}
        x, & \text{if } \theta = w,\\
        1, & \text{if } \theta = b_0
    \end{cases}
$$

Gradient to parameter $\theta$ at hidden layer could be defined as:
$$
\begin{align*}
\frac{\partial L}{\partial \theta} &= \frac{\partial L}{\partial y_{out}} \frac{\partial y_{out}}{\partial y_{in}} \frac{ \partial y_{in} }{ \partial \theta }  \\
 & = (\hat{y}-y) \cdot \Omega
\end{align*}
$$

And again, $\Omega$ depends whether observing input biases or weight biases, such as:

$$
\Omega = 
    \begin{cases}
        z, & \text{if } \theta = v,\\
        1, & \text{if } \theta = b_1
    \end{cases}
$$

In [17]:
def backpropagation(x_i, y, t_i, z_in, z_out, w, b0, v, b1, lr):
    
    delta_out = BCE_deriv(t_i, y)
    
    dv = z_out.reshape(-1, 1) * delta_out
    db1 = delta_out

    delta_hidden = (v.flatten() * delta_out) * relu_deriv(z_in)
    
    dw = np.outer(x_i, delta_hidden)
    db0 = delta_hidden
        
    v  -= lr * dv
    b1 -= lr * db1
    w  -= lr * dw
    b0 -= lr * db0

    return w, b0, v, b1

## Training

In [18]:
max_epochs = 10

start = time.time()
for epoch in range(max_epochs):
    total_loss = 0
    
    for i in range(len(x)):
        z_in = x[i] @ w + b0
        z_out = relu(z_in)
        y_in = z_out @ v + b1
        y = sigmoid(y_in)

        loss = BCE(t[i], y)
        total_loss += loss

        w, b0, v, b1 = backpropagation(x[i], y, t[i], z_in, z_out, w, b0, v, b1, lr)

    print(f'average loss epoch {epoch+1}: {total_loss / len(x)}')
    
end = (time.time() - start) / 60
print(f'\ntraining time: {end:.2f} minutes')

average loss epoch 1: [0.12293773]
average loss epoch 2: [0.21371705]
average loss epoch 3: [0.16827401]
average loss epoch 4: [0.14235538]
average loss epoch 5: [0.12402579]
average loss epoch 6: [0.10989028]
average loss epoch 7: [0.10080692]
average loss epoch 8: [0.09319838]
average loss epoch 9: [0.08743958]
average loss epoch 10: [0.08275613]

training time: 23.67 minutes


In [19]:
params

{'w': array([[ 0.01017873,  0.03685942,  0.0228473 , ...,  0.00147089,
         -0.01262457, -0.00658382],
        [ 0.02214439,  0.02673515, -0.01710286, ..., -0.01942932,
          0.02255407, -0.02467913],
        [-0.01427463, -0.00464749, -0.00465425, ..., -0.01330469,
          0.0294181 , -0.03812609],
        ...,
        [-0.0289793 ,  0.01513426, -0.02534047, ...,  0.02406434,
         -0.0250462 ,  0.00182167],
        [-0.00558618,  0.0131706 ,  0.00157024, ..., -0.00065312,
         -0.03689742,  0.00984853],
        [-0.00526775, -0.00144777,  0.02459505, ..., -0.00826443,
          0.01796811, -0.01003803]], shape=(34609, 128)),
 'b0': array([ 1.52011618e-03,  2.19889157e-02,  6.73232006e-04,  7.01144051e-04,
        -1.84503424e-03, -3.39216273e-04,  2.73503636e-03,  1.30793505e-03,
        -7.78417385e-04,  5.27184363e-03, -7.21472735e-04,  1.21977484e-02,
         5.25212296e-03,  7.85288939e-03, -5.30821075e-04,  1.01535314e-02,
         1.48356126e-02,  3.42850257e-

# V. Inference

## Cleaning Text and Encoding BoW for Data Test

In [20]:
test_cleaned = X_test.copy()
for i in range(len(X_test)):
    test_cleaned.iloc[i] = preprotext(test_cleaned.iloc[i])

In [21]:
test_bow = []
for tokens in test_cleaned:
    tokens = build_bow(tokens, vocab)
    test_bow.append(tokens)

## Prediction

In [22]:
xt = np.array(test_bow)
y_preds = []
    
for i in range(len(xt)):
    z_in = xt[i] @ w + b0
    z_out = relu(z_in)
    y_in = z_out @ v + b1
    y = sigmoid(y_in)
    y_preds.append(y)

y_preds = np.array(y_preds)

In [23]:
y_hat = (y_preds >= 0.5).astype(int)
accuracy = round((y_hat.flatten() == y_test).mean(), 4) * 100

In [24]:
accuracy

np.float64(99.83)

# VI. Discussion

Although the test accuracy reached 99.56%, I could say the model overfits and this accuracy can be concluded false. To observe tokens in data test appeared in vocabulary, we may use:

In [25]:
for i in range(len(xt)):
    if xt[0][i] != 0:
        print(i)

324
681
813


From the results above, we can conclude that Vanilla BoW did not work well to the dataset. This is primarily because BoW splits words independently without context. Further improvement would be made with TF-IDF and Embeddings.

# Improvement with TF-IDF

In [26]:
def calcTermFrequency(documents):
    termFreqList = []
    for document in documents:
        tf_doc = {} # term frequency for every documents
        for token in document:
            tf_doc[token] = tf_doc.get(token, 0) + 1
        termFreqList.append(tf_doc)
    return termFreqList

def calcDocFrequency(termFreqList):
    df = {}
    for tf_doc in termFreqList:
        for token in tf_doc.keys():
            df[token] = df.get(token, 0) + 1
    return df

def calcIDF(N, docFreq):
    idfList = {}
    for df in docFreq:
        idf_df = np.log10(N/docFreq.get(df))
        idfList.update({df: idf_df})
    return idfList

def calcTFIDF(termFreq, docFreq, idf):
    tf_idf = [] 
    for tf in termFreq:
        tf_idf_doc = {}
        for tokens in tf:
            tf_idf_doc[tokens] = tf[tokens] * idf[tokens]
        tf_idf.append(tf_idf_doc)
    return tf_idf

In [27]:
tf_train = calcTermFrequency(train_cleaned)
tf_train

[{'subject': 1,
  'naturally': 1,
  'irresistible': 1,
  'your': 7,
  'corporate': 1,
  'identity': 1,
  'lt': 1,
  'is': 3,
  'really': 1,
  'hard': 1,
  'to': 4,
  'recollect': 1,
  'a': 5,
  'company': 3,
  'the': 5,
  'market': 2,
  'full': 1,
  'of': 4,
  'suqgestions': 1,
  'and': 5,
  'information': 1,
  'isoverwhelminq': 1,
  'but': 2,
  'good': 2,
  'catchy': 1,
  'logo': 3,
  'stylish': 1,
  'statlonery': 1,
  'outstanding': 1,
  'website': 2,
  'will': 6,
  'make': 2,
  'task': 1,
  'much': 2,
  'easier': 1,
  'we': 3,
  'do': 2,
  'not': 2,
  'promise': 2,
  'that': 3,
  'havinq': 1,
  'ordered': 1,
  'iogo': 1,
  'automaticaily': 1,
  'become': 2,
  'world': 1,
  'ieader': 1,
  'it': 2,
  'isguite': 1,
  'ciear': 1,
  'without': 1,
  'products': 1,
  'effective': 2,
  'business': 2,
  'organization': 1,
  'practicable': 1,
  'aim': 1,
  'be': 2,
  'hotat': 1,
  'nowadays': 1,
  'marketing': 2,
  'efforts': 1,
  'more': 1,
  'here': 1,
  'list': 1,
  'clear': 1,
  'benefits

In [28]:
df_train = calcDocFrequency(tf_train)
df_train

{'subject': 4582,
 'naturally': 13,
 'irresistible': 4,
 'your': 2670,
 'corporate': 180,
 'identity': 81,
 'lt': 27,
 'is': 3121,
 'really': 267,
 'hard': 147,
 'to': 4148,
 'recollect': 25,
 'a': 3496,
 'company': 555,
 'the': 4023,
 'market': 461,
 'full': 303,
 'of': 3396,
 'suqgestions': 14,
 'and': 3608,
 'information': 1019,
 'isoverwhelminq': 7,
 'but': 1010,
 'good': 671,
 'catchy': 24,
 'logo': 105,
 'stylish': 14,
 'statlonery': 15,
 'outstanding': 78,
 'website': 297,
 'will': 2279,
 'make': 804,
 'task': 53,
 'much': 512,
 'easier': 69,
 'we': 2412,
 'do': 1304,
 'not': 1774,
 'promise': 48,
 'that': 2347,
 'havinq': 9,
 'ordered': 57,
 'iogo': 22,
 'automaticaily': 5,
 'become': 165,
 'world': 235,
 'ieader': 16,
 'it': 2167,
 'isguite': 13,
 'ciear': 9,
 'without': 261,
 'products': 251,
 'effective': 134,
 'business': 786,
 'organization': 139,
 'practicable': 24,
 'aim': 37,
 'be': 2582,
 'hotat': 23,
 'nowadays': 25,
 'marketing': 210,
 'efforts': 163,
 'more': 1040,


In [29]:
idf_train = calcIDF(train_cleaned.shape[0], df_train)
idf_train

{'subject': np.float64(0.0),
 'naturally': np.float64(2.5471117325465418),
 'irresistible': np.float64(3.058995093525416),
 'your': np.float64(0.23454382348880348),
 'corporate': np.float64(1.4057825797500727),
 'identity': np.float64(1.752570065974729),
 'lt': np.float64(2.2296913206943914),
 'is': np.float64(0.16676131618804604),
 'really': np.float64(1.2345438234888035),
 'hard': np.float64(1.4937377501052027),
 'to': np.float64(0.04321633713637533),
 'recollect': np.float64(2.2631150761813412),
 'a': np.float64(0.11748366089101331),
 'company': np.float64(0.9167621017307025),
 'the': np.float64(0.05650505228211735),
 'market': np.float64(0.9973541594637305),
 'full': np.float64(1.1796124563510737),
 'of': np.float64(0.13008740328146365),
 'suqgestions': np.float64(2.5149270491751405),
 'and': np.float64(0.10378855598347461),
 'information': np.float64(0.6528809008469524),
 'isoverwhelminq': np.float64(2.815957044839122),
 'but': np.float64(0.6567337110707361),
 'good': np.float64(0

In [30]:
tf_idf_train = calcTFIDF(tf_train, df_train, idf_train)
tf_idf_train

[{'subject': np.float64(0.0),
  'naturally': np.float64(2.5471117325465418),
  'irresistible': np.float64(3.058995093525416),
  'your': np.float64(1.6418067644216243),
  'corporate': np.float64(1.4057825797500727),
  'identity': np.float64(1.752570065974729),
  'lt': np.float64(2.2296913206943914),
  'is': np.float64(0.5002839485641382),
  'really': np.float64(1.2345438234888035),
  'hard': np.float64(1.4937377501052027),
  'to': np.float64(0.1728653485455013),
  'recollect': np.float64(2.2631150761813412),
  'a': np.float64(0.5874183044550665),
  'company': np.float64(2.7502863051921076),
  'the': np.float64(0.28252526141058676),
  'market': np.float64(1.994708318927461),
  'full': np.float64(1.1796124563510737),
  'of': np.float64(0.5203496131258546),
  'suqgestions': np.float64(2.5149270491751405),
  'and': np.float64(0.518942779917373),
  'information': np.float64(0.6528809008469524),
  'isoverwhelminq': np.float64(2.815957044839122),
  'but': np.float64(1.3134674221414722),
  'goo

## Training with TF-IDF

In [31]:
word2idx = {word: i for i, word in enumerate(vocab)}
word2idx

{'rosia': 0,
 'exorbitant': 1,
 'skies': 2,
 'intmail': 3,
 'sandwiches': 4,
 'ubs': 5,
 'lyczak': 6,
 'includ': 7,
 'vb': 8,
 'cbot': 9,
 'surbl': 10,
 'sizable': 11,
 'disseminated': 12,
 'whirligig': 13,
 'nasz': 14,
 'burnett': 15,
 'betty': 16,
 'larrissa': 17,
 'choosy': 18,
 'factory': 19,
 'highiy': 20,
 'erratic': 21,
 'corpcustserv': 22,
 'geologize': 23,
 'summing': 24,
 'kmart': 25,
 'redhill': 26,
 'scalp': 27,
 'publishers': 28,
 'verbal': 29,
 'unobserved': 30,
 'gisele': 31,
 'rtun': 32,
 '4869': 33,
 '5914': 34,
 'attacks': 35,
 'plots': 36,
 'grid': 37,
 'circa': 38,
 'kr': 39,
 'bobbie': 40,
 'derring': 41,
 'fibre': 42,
 'efellows': 43,
 'philipp': 44,
 'experienced': 45,
 'desperate': 46,
 'aii': 47,
 'simplicity': 48,
 'assurances': 49,
 'folders': 50,
 'dundalk': 51,
 'mcclure': 52,
 'bump': 53,
 'dunnaway': 54,
 'parra': 55,
 'dispatches': 56,
 'igniting': 57,
 'hedgeing': 58,
 'noticed': 59,
 'brooke': 60,
 'calger': 61,
 'appliction': 62,
 'alda': 63,
 'avail'

In [32]:
def tfidf_to_vector(tfidf_doc, vocab):
    vec = np.zeros(len(vocab))
    for token, value in tfidf_doc.items():
        if token in vocab:
            vec[vocab[token]] = value
    return vec

In [33]:
x = np.array([
    tfidf_to_vector(doc, word2idx)
    for doc in tf_idf_train
])

x

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(4582, 34609))

In [34]:
x.shape[0]

4582

In [35]:
t = y_train # target
w = np.random.randn(x.shape[1], H) * np.sqrt(2/x.shape[0]) # He initialization
b0 = np.zeros(H) # hidden bias
v = np.random.randn(H, 1) * np.sqrt(1/H) # Xavier initialization
b1 = np.zeros(1) # output bias

In [36]:
w, b0, v, b1

(array([[ 0.00963774,  0.00956899, -0.02308501, ..., -0.01817165,
          0.01185585,  0.000838  ],
        [-0.00326064,  0.03497057,  0.01599232, ..., -0.00846482,
          0.00456194, -0.0034937 ],
        [ 0.01714601, -0.04870086, -0.00471728, ..., -0.00046645,
          0.00776851, -0.00503743],
        ...,
        [ 0.03175781,  0.02874857,  0.00894158, ..., -0.01887936,
          0.00096321,  0.01927698],
        [ 0.00655183,  0.00193838, -0.01376244, ..., -0.0042251 ,
          0.00042141, -0.03847339],
        [ 0.0176827 , -0.02888621, -0.034043  , ...,  0.03641602,
         -0.0057478 ,  0.01064742]], shape=(34609, 128)),
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

In [37]:
max_epochs = 10

start = time.time()
for epoch in range(max_epochs):
    total_loss = 0
    
    for i in range(len(x)):
        z_in = x[i] @ w + b0
        z_out = relu(z_in)
        y_in = z_out @ v + b1
        y = sigmoid(y_in)

        loss = BCE(t[i], y)
        total_loss += loss

        w, b0, v, b1 = backpropagation(x[i], y, t[i], z_in, z_out, w, b0, v, b1, lr)

    print(f'average loss epoch {epoch+1}: {total_loss / len(x)}')
    
end = (time.time() - start) / 60
print(f'\ntraining time: {end:.2f} minutes')

average loss epoch 1: [0.14950142]
average loss epoch 2: [0.22088605]
average loss epoch 3: [0.20369657]
average loss epoch 4: [0.18564135]
average loss epoch 5: [0.1680535]
average loss epoch 6: [0.15193855]
average loss epoch 7: [0.13768414]
average loss epoch 8: [0.12526211]
average loss epoch 9: [0.11440031]
average loss epoch 10: [0.1049019]

training time: 21.63 minutes


In [38]:
w, b0, v, b1

(array([[ 0.00961279,  0.00950059, -0.02308501, ..., -0.01815246,
          0.01180192,  0.00088695],
        [-0.00326064,  0.03497832,  0.01598567, ..., -0.00846482,
          0.00456195, -0.0034937 ],
        [ 0.01713053, -0.04877349, -0.00468037, ..., -0.00046645,
          0.00776851, -0.00499363],
        ...,
        [ 0.03175781,  0.02874858,  0.00894158, ..., -0.01887936,
          0.00096321,  0.01927697],
        [ 0.00658179,  0.00197656, -0.01376244, ..., -0.0042251 ,
          0.00052187, -0.03849838],
        [ 0.0176827 , -0.02885106, -0.034043  , ...,  0.03639651,
         -0.0057436 ,  0.01064742]], shape=(34609, 128)),
 array([ 2.58200320e-03,  1.25610805e-02, -2.20000990e-03, -3.47722893e-04,
         1.53469506e-04,  1.53064557e-03,  4.98013796e-02,  1.47953554e-02,
        -2.45007086e-03, -6.87235675e-04, -3.92452770e-03,  1.12795969e-03,
         8.27372880e-04,  3.31611440e-02,  2.21976727e-02,  1.99211288e-02,
         2.04389228e-03, -1.51305865e-03,  1.7806

## Testing

In [39]:
tf_test = calcTermFrequency(test_cleaned)
df_test = calcDocFrequency(tf_test)
idf_test = calcIDF(test_cleaned.shape[0], df_test)
tf_idf_test = calcTFIDF(tf_test, df_test, idf_test)

In [40]:
xt = np.array([
    tfidf_to_vector(doc, word2idx)
    for doc in tf_idf_test
])

In [41]:
y_preds = []
    
for i in range(len(xt)):
    z_in = xt[i] @ w + b0
    z_out = relu(z_in)
    y_in = z_out @ v + b1
    y = sigmoid(y_in)
    y_preds.append(y)

y_preds = np.array(y_preds)

In [42]:
y_hat = (y_preds >= 0.5).astype(int)
accuracy = round((y_hat.flatten() == y_test).mean(), 4) * 100
accuracy

np.float64(99.21)

# Improvement with Word Embeddings

In [43]:
"""TODO(): implement improvements with Word Embeddings"""

'TODO(): implement improvements with Word Embeddings'